
# Data Quality Checks

This notebook validates the analytical dataset before further data processing and statistical analysis.

Main steps:
- Inspect dataset dimensions and structure
- Check the county-year key for duplicate observations
- Verify the balanced panel structure
- Identify and assess missing values
- Validate data types and unexpected negative or zero values
- Review descriptive statistics and potential outliers
- Check the plausibility of crop yield and climate variables
- Summarize the overall quality and consistency of the dataset

In [2]:
import sys
!{sys.executable} -m pip install pandas numpy


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd

In [11]:
df = pd.read_csv("../../data/processed/analysis_dataset_ins.csv")
df.head(2)

,an,judet,emig_masc_nr,emig_fem_nr,emigranti_total_nr,emig_15_64_nr,imig_masc_nr,imig_fem_nr,imigranti_total_nr,imigranti_15_64_nr,...,z_rezid_combine,indice_mecanizare_rezidual,schimbare_ocupati_agri_nr,schimbare_ocupati_agri_pct,schimbare_pondere_ocupati_agri,ocupati_agri_1000ha,migratie_neta_total_nr,rata_migratie_neta_total,migratie_neta_15_64_nr,rata_migratie_neta_15_64
0,2012,Alba,143,144,287,222,43,33,76,66,...,0.802190,0.325906,NaN,NaN,NaN,772.194749,-211,-0.619597,-156,-0.679031
1,2013,Alba,104,158,262,224,53,41,94,83,...,0.841744,0.412092,-2400.0,-4.848485,-1.451137,746.339608,-168,-0.496060,-141,-0.618020


In [10]:
df.shape

(451, 82)

In [5]:
for col in df.columns:
    print(col)

an
judet
emig_masc_nr
emig_fem_nr
emigranti_total_nr
emig_15_64_nr
imig_masc_nr
imig_fem_nr
imigranti_total_nr
imigranti_15_64_nr
populatie_15_64
populatie_rurala
populatie_totala
pondere_rurala
rata_emig_def
rata_imig_def
rata_emig_15_64
rata_imig_15_64
intensitate_migratie
sup_grau_ha
sup_orz_orzoaica_ha
sup_porumb_boabe_ha
sup_floarea_soarelui_ha
sup_rapita_ha
sup_soia_boabe_ha
sup_totala_cultivata_ha
pondere_grau
pondere_orz_orzoaica
pondere_porumb_boabe
pondere_floarea_soarelui
pondere_rapita
pondere_soia_boabe
prod_grau_tone
prod_orz_orzoaica_tone
prod_porumb_boabe_tone
prod_floarea_soarelui_tone
prod_rapita_tone
prod_soia_boabe_tone
yield_grau
yield_orz_orzoaica
yield_porumb_boabe
yield_floarea_soarelui
yield_rapita
yield_soia_boabe
ocupati_total_nr
ocupati_agri_nr
pondere_ocupati_agri
tractoare_nr
pluguri_nr
semanatori_mecanice_nr
masini_stropit_prafuit_mecanice_nr
combine_cereale_nr
tractoare_1000ha
pluguri_1000ha
semanatori_1000ha
masini_stropit_1000ha
combine_1000ha
z_tracto

In [13]:
# verific cheia judet-an
df.duplicated(subset=["judet", "an"]).sum()

np.int64(0)

In [9]:
df["judet"].nunique()


41

In [10]:
sorted(df["an"].unique())

[np.int64(2012),
 np.int64(2013),
 np.int64(2014),
 np.int64(2015),
 np.int64(2016),
 np.int64(2017),
 np.int64(2018),
 np.int64(2019),
 np.int64(2020),
 np.int64(2021),
 np.int64(2022)]

In [12]:
# dimensiune panel

n_judete = df["judet"].nunique()
n_ani = df["an"].nunique()
n_asteptat = n_judete * n_ani

print("Județe:", n_judete)
print("Ani:", n_ani)
print("Observații așteptate:", n_asteptat)
print("Observații existente:", len(df))

Județe: 41
Ani: 11
Observații așteptate: 451
Observații existente: 451


In [14]:
# missing values
df.isna().sum().sort_values(ascending=False)

yield_soia_boabe                  50
schimbare_pondere_ocupati_agri    41
yield_rapita                      27
yield_floarea_soarelui            14
emigranti_total_nr                 0
                                  ..
rezid_semanatori                   0
log_combine                        0
rezid_combine                      0
z_rezid_combine                    0
indice_mecanizare_rezidual         0
Length: 82, dtype: int64

In [15]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

yield_soia_boabe                  50
schimbare_pondere_ocupati_agri    41
yield_rapita                      27
yield_floarea_soarelui            14
dtype: int64

In [16]:
missing_pct = df.isna().mean() * 100
missing_pct[missing_pct > 0].sort_values(ascending=False)

yield_soia_boabe                  11.086475
schimbare_pondere_ocupati_agri     9.090909
yield_rapita                       5.986696
yield_floarea_soarelui             3.104213
dtype: float64

In [18]:
# tipuri date
df.dtypes

an                                  int64
judet                              object
emig_masc_nr                        int64
emig_fem_nr                         int64
emigranti_total_nr                  int64
                                   ...   
log_combine                       float64
rezid_combine                     float64
z_rezid_combine                   float64
indice_mecanizare_rezidual        float64
schimbare_pondere_ocupati_agri    float64
Length: 82, dtype: object

In [19]:
# valori negative
vars_nonnegative = [
    col for col in df.columns
    if any(x in col for x in ["nr", "sup_", "prod_", "yield_", "precipitatii", "tractoare", "pluguri", "semanatori", "combine"])
]

negatives = {}

for col in vars_nonnegative:
    if pd.api.types.is_numeric_dtype(df[col]):
        n_neg = (df[col] < 0).sum()
        if n_neg > 0:
            negatives[col] = n_neg

negatives

{'z_tractoare_1000ha': np.int64(313),
 'z_pluguri_1000ha': np.int64(300),
 'z_semanatori_1000ha': np.int64(266),
 'z_combine_1000ha': np.int64(282),
 'schimbare_precipitatii_anuala_mm': np.int64(213),
 'schimbare_precipitatii_anuala_pct': np.int64(213),
 'rezid_tractoare': np.int64(225),
 'z_rezid_tractoare': np.int64(225),
 'rezid_pluguri': np.int64(237),
 'z_rezid_pluguri': np.int64(237),
 'rezid_semanatori': np.int64(222),
 'z_rezid_semanatori': np.int64(222),
 'rezid_combine': np.int64(238),
 'z_rezid_combine': np.int64(238)}

In [20]:
# valori 0

zero_counts = {}

for col in vars_nonnegative:
    if pd.api.types.is_numeric_dtype(df[col]):
        n_zero = (df[col] == 0).sum()
        if n_zero > 0:
            zero_counts[col] = n_zero

zero_counts

{'sup_floarea_soarelui_ha': np.int64(14),
 'sup_rapita_ha': np.int64(27),
 'sup_soia_boabe_ha': np.int64(50),
 'prod_floarea_soarelui_tone': np.int64(14),
 'prod_rapita_tone': np.int64(27),
 'prod_soia_boabe_tone': np.int64(51),
 'yield_soia_boabe': np.int64(2)}

In [22]:
# valori extreme

vars_key = [
    "rata_emig_15_64",
    "rata_imig_15_64",
    "intensitate_migratie",
    "ocupati_agri_nr",
    "pondere_ocupati_agri",
    "indice_mecanizare_vechi",
    "indice_mecanizare_rezidual",
    "yield_grau",
    "yield_orz_orzoaica",
    "yield_porumb_boabe",
    "yield_floarea_soarelui",
    "yield_rapita",
    "yield_soia_boabe",
    "temperatura_medie_anuala_C",
    "precipitatii_anuale_mm"
]

df[vars_key].describe().T

,count,mean,std,min,25%,50%,75%,max
rata_emig_15_64,451.0,1.376333e+00,0.960898,0.153051,0.767320,1.108827,1.748389,8.306831
rata_imig_15_64,451.0,2.140644e+00,5.530722,0.072290,0.300564,0.471763,0.911199,46.459290
intensitate_migratie,451.0,2.645393e+00,4.282977,0.283551,0.898064,1.365764,2.231161,35.844221
ocupati_agri_nr,451.0,4.330222e+04,20009.888543,9100.000000,28150.000000,40700.000000,56200.000000,112300.000000
pondere_ocupati_agri,451.0,2.663657e+01,11.072896,5.043341,18.384394,25.440000,34.409606,55.906507
indice_mecanizare_vechi,451.0,1.773836e-11,0.930987,-0.980996,-0.683922,-0.354697,0.327038,3.856129
indice_mecanizare_rezidual,451.0,-1.969353e-18,0.907752,-2.530997,-0.646172,0.061402,0.674135,2.082357
yield_grau,451.0,3.774817e+00,0.884442,0.983083,3.195169,3.777532,4.401597,5.994879
yield_orz_orzoaica,451.0,3.240934e+00,0.943051,1.017149,2.605668,3.093537,3.856614,7.219444
yield_porumb_boabe,451.0,4.631103e+00,1.719827,0.252296,3.465615,4.476385,5.786776,10.420537


In [23]:
# variabile lipsa verificari
df[df["yield_soia_boabe"].isna()][
    ["judet", "an", "sup_soia_boabe_ha", "prod_soia_boabe_tone", "yield_soia_boabe"]
].head(20)

,judet,an,sup_soia_boabe_ha,prod_soia_boabe_tone,yield_soia_boabe
23,Arges,2013,0,0,NaN
99,Buzau,2012,0,0,NaN
100,Buzau,2013,0,0,NaN
101,Buzau,2014,0,0,NaN
111,Caras-Severin,2012,0,0,NaN
113,Caras-Severin,2013,0,0,NaN
115,Caras-Severin,2014,0,0,NaN
165,Dambovita,2012,0,0,NaN
167,Dambovita,2013,0,0,NaN
168,Dolj,2013,0,0,NaN


In [24]:
df[df["yield_rapita"].isna()][
    ["judet", "an", "sup_rapita_ha", "prod_rapita_tone", "yield_rapita"]
].head(20)

,judet,an,sup_rapita_ha,prod_rapita_tone,yield_rapita
55,Bistrita-Nasaud,2012,0,0,NaN
59,Bistrita-Nasaud,2016,0,0,NaN
61,Bistrita-Nasaud,2018,0,0,NaN
62,Bistrita-Nasaud,2019,0,0,NaN
209,Gorj,2012,0,0,NaN
210,Gorj,2013,0,0,NaN
212,Gorj,2015,0,0,NaN
213,Gorj,2016,0,0,NaN
214,Gorj,2017,0,0,NaN
216,Gorj,2019,0,0,NaN


In [25]:
df[df["yield_floarea_soarelui"].isna()][
    ["judet", "an", "sup_floarea_soarelui_ha", "prod_floarea_soarelui_tone", "yield_floarea_soarelui"]
].head(20)

,judet,an,sup_floarea_soarelui_ha,prod_floarea_soarelui_tone,yield_floarea_soarelui
78,Brasov,2012,0,0,NaN
80,Brasov,2013,0,0,NaN
82,Brasov,2014,0,0,NaN
154,Covasna,2012,0,0,NaN
155,Covasna,2013,0,0,NaN
220,Harghita,2012,0,0,NaN
221,Harghita,2013,0,0,NaN
222,Harghita,2014,0,0,NaN
223,Harghita,2015,0,0,NaN
224,Harghita,2016,0,0,NaN


In [29]:
yield_cols = [
    "yield_grau",
    "yield_orz_orzoaica",
    "yield_porumb_boabe",
    "yield_floarea_soarelui",
    "yield_rapita",
    "yield_soia_boabe"
]

df[yield_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
yield_grau,451.0,3.774817,0.884442,0.983083,3.195169,3.777532,4.401597,5.994879
yield_orz_orzoaica,451.0,3.240934,0.943051,1.017149,2.605668,3.093537,3.856614,7.219444
yield_porumb_boabe,451.0,4.631103,1.719827,0.252296,3.465615,4.476385,5.786776,10.420537
yield_floarea_soarelui,437.0,2.133286,0.632567,0.427039,1.696705,2.137745,2.532534,3.916080
yield_rapita,424.0,2.341264,0.605324,0.583333,1.951174,2.403778,2.753288,4.029930
yield_soia_boabe,401.0,1.968609,0.710654,0.000000,1.461233,1.916667,2.426165,4.612922


In [30]:
df[["judet", "an", "rata_imig_15_64", "imigranti_15_64_nr", "populatie_15_64"]] \
    .sort_values("rata_imig_15_64", ascending=False) \
    .head(10)

,judet,an,rata_imig_15_64,imigranti_15_64_nr,populatie_15_64
433,Vaslui,2019,46.459290,10776,231945
431,Vaslui,2018,42.817246,10088,235606
439,Vaslui,2022,39.290230,9105,231737
429,Vaslui,2017,37.549481,9002,239737
423,Vaslui,2014,29.806073,7399,248238
255,Iasi,2014,23.985504,12747,531446
427,Vaslui,2016,23.470164,5709,243245
437,Vaslui,2021,21.967860,5036,229244
421,Vaslui,2013,21.721806,5397,248460
72,Botosani,2018,21.711763,5312,244660


In [31]:
df[["judet", "an", "intensitate_migratie", "rata_emig_15_64", "rata_imig_15_64"]] \
    .sort_values("intensitate_migratie", ascending=False) \
    .head(10)

,judet,an,intensitate_migratie,rata_emig_15_64,rata_imig_15_64
433,Vaslui,2019,35.844221,3.065382,46.459290
439,Vaslui,2022,34.166154,8.306831,39.290230
431,Vaslui,2018,31.043486,2.563602,42.817246
429,Vaslui,2017,26.349127,1.814488,37.549481
437,Vaslui,2021,20.904891,4.920521,21.967860
423,Vaslui,2014,19.856818,0.801650,29.806073
263,Iasi,2022,19.464319,7.783769,16.828434
197,Galati,2022,18.789401,6.018217,19.243669
255,Iasi,2014,17.927698,0.848628,23.985504
427,Vaslui,2016,16.691375,1.870542,23.470164


In [36]:
df[["judet","an", "yield_orz_orzoaica"]].sort_values("yield_orz_orzoaica", ascending=False).head(10)

,judet,an,yield_orz_orzoaica
131,Caras-Severin,2022,7.219444
122,Calarasi,2018,7.148635
196,Galati,2021,5.866553
402,Timis,2018,5.866207
248,Ialomita,2018,5.794082
251,Ialomita,2021,5.649341
120,Calarasi,2017,5.645996
405,Timis,2021,5.613850
124,Calarasi,2019,5.552983
247,Ialomita,2017,5.439409


In [38]:
df[["judet","an", "yield_soia_boabe"]].sort_values("yield_soia_boabe", ascending=True).head(10)

,judet,an,yield_soia_boabe
212,Gorj,2015,0.000000
166,Dolj,2012,0.000000
385,Teleorman,2012,0.056391
55,Bistrita-Nasaud,2012,0.545455
333,Prahova,2015,0.561250
32,Arges,2022,0.562500
363,Sibiu,2012,0.563131
291,Mehedinti,2017,0.575000
0,Alba,2012,0.618421
341,Salaj,2012,0.666667


In [39]:
df[df["yield_soia_boabe"] == 0][
    ["judet", "an", "sup_soia_boabe_ha", "prod_soia_boabe_tone", "yield_soia_boabe"]
]

,judet,an,sup_soia_boabe_ha,prod_soia_boabe_tone,yield_soia_boabe
166,Dolj,2012,63,0,0.0
212,Gorj,2015,42,0,0.0


In [40]:
df[["judet", "an", "precipitatii_anuale_mm"]] \
    .sort_values("precipitatii_anuale_mm") \
    .head(10)

,judet,an,precipitatii_anuale_mm
97,Braila,2022,251.774761
417,Tulcea,2022,267.930495
197,Galati,2022,282.090799
252,Ialomita,2022,285.179213
130,Calarasi,2022,289.993956
153,Constanta,2022,315.453109
414,Tulcea,2019,344.080609
439,Vaslui,2022,349.786485
274,Ilfov,2022,351.509136
208,Giurgiu,2022,357.476500


In [41]:
df[["judet", "an", "precipitatii_anuale_mm"]] \
    .sort_values("precipitatii_anuale_mm", ascending=False) \
    .head(10)

,judet,an,precipitatii_anuale_mm
235,Hunedoara,2016,1260.069888
211,Gorj,2014,1254.282987
288,Mehedinti,2014,1215.405180
115,Caras-Severin,2014,1188.065857
422,Valcea,2014,1168.313136
284,Maramures,2021,1166.374369
119,Caras-Severin,2016,1162.242463
367,Sibiu,2016,1161.599790
280,Maramures,2017,1155.820171
426,Valcea,2016,1130.664972


In [42]:
df.groupby("judet")["an"].nunique().sort_values()

judet
Alba               11
Arad               11
Arges              11
Bacau              11
Bihor              11
Bistrita-Nasaud    11
Botosani           11
Braila             11
Brasov             11
Buzau              11
Calarasi           11
Caras-Severin      11
Cluj               11
Constanta          11
Covasna            11
Dambovita          11
Dolj               11
Galati             11
Giurgiu            11
Gorj               11
Harghita           11
Hunedoara          11
Ialomita           11
Iasi               11
Ilfov              11
Maramures          11
Mehedinti          11
Mures              11
Neamt              11
Olt                11
Prahova            11
Salaj              11
Satu Mare          11
Sibiu              11
Suceava            11
Teleorman          11
Timis              11
Tulcea             11
Valcea             11
Vaslui             11
Vrancea            11
Name: an, dtype: int64

In [43]:
df[["judet", "an", "rata_imig_15_64", "imigranti_15_64_nr", "populatie_15_64"]] \
    .sort_values("rata_imig_15_64", ascending=False) \
    .head(10)

,judet,an,rata_imig_15_64,imigranti_15_64_nr,populatie_15_64
433,Vaslui,2019,46.459290,10776,231945
431,Vaslui,2018,42.817246,10088,235606
439,Vaslui,2022,39.290230,9105,231737
429,Vaslui,2017,37.549481,9002,239737
423,Vaslui,2014,29.806073,7399,248238
255,Iasi,2014,23.985504,12747,531446
427,Vaslui,2016,23.470164,5709,243245
437,Vaslui,2021,21.967860,5036,229244
421,Vaslui,2013,21.721806,5397,248460
72,Botosani,2018,21.711763,5312,244660


## Data Quality Summary

- The dataset forms a balanced panel of 41 Romanian counties observed over 11 years (2012–2022), resulting in 451 observations.
- No duplicate county-year observations were identified.
- Missing values are limited and primarily related to crop yields where cultivated area or production is unavailable or zero.
- Negative values occur only in variables where they are theoretically valid, such as standardized scores, residuals, and annual changes.
- Crop yield and climate variables were checked for plausibility.
- Potential migration outliers were inspected individually before further analysis.